# Acto 2 — Synthetic Offshore Network (sintético)

Embeddings + HNSW + path queries cuantificadas + reducers + path-anomaly. ~50 min.

Lee la narrativa completa en [`docs/tutorial/acto_2_synthetic_offshore/README.md`](../../docs/tutorial/acto_2_synthetic_offshore/README.md). Las queries NQL se cargan desde `docs/tutorial/acto_2_synthetic_offshore/queries/*.nql`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

TUTORIALS_DIR = Path.cwd().resolve()
if TUTORIALS_DIR.name == "notebooks":
    TUTORIALS_DIR = TUTORIALS_DIR.parent
if str(TUTORIALS_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIALS_DIR))

import nopaldb
from shared import REPO_ROOT, db_path, load_nql
from shared.viz import execute_to_df
from shared.embeddings import attach_node_embeddings, encode_texts

DB = db_path("synthetic_offshore")
GENERATOR = TUTORIALS_DIR / "shared" / "synthetic_offshore_dataset.py"
print("DB target:", DB)

## 1. Generar DB si no existe

In [ ]:
import subprocess

if not Path(DB).exists():
    print(f"Generando {DB}...")
    subprocess.run(
        [sys.executable, str(GENERATOR), "--db", DB, "--reset"],
        cwd=str(REPO_ROOT),
        check=True,
    )
else:
    print(f"Reusando DB existente: {DB}")

graph = nopaldb.Graph.open(DB)
stats = graph.get_stats()
print(f"total_nodes: {int(stats['total_nodes'])}")
print(f"total_edges: {int(stats['total_edges'])}")

## 2. Paso 1 — Schema discovery: ¿qué hay en este dataset?

**¿Qué es el "schema"?** Es la estructura del dataset: cuántos tipos de entidades existen y cuántas hay de cada tipo. Antes de hacer cualquier consulta útil, necesitas saber con qué estás trabajando.

En una investigación real de fraude financiero, nunca recibes un manual que diga "hay 52 empresas offshore y 50 directivos". La primera pregunta siempre es: *¿qué hay aquí?*

In [ ]:
df_schema = execute_to_df(graph, load_nql("acto_2", "01_schema_discovery.nql"))
df_schema

## 3. Paso 2 — Índice hash: búsqueda instantánea por nombre

**¿Qué es un índice?** Imagina buscar un número en una guía telefónica sin índice: tendrías que leer todas las páginas. Con índice, vas directo a la letra correcta. Un *Hash index* hace algo similar: convierte el nombre en una dirección matemática y salta directo a ella en O(1) — tiempo constante sin importar cuántos registros haya.

El generador creó un Hash index sobre `OffshoreEntity(name)`. La siguiente consulta lo usa automáticamente — el planificador de NQL detecta que el `WHERE` coincide con el índice y evita escanear todas las entidades.

In [ ]:
df_lookup = execute_to_df(graph, load_nql("acto_2", "02_indices.nql"))
df_lookup

## 4. Paso 3 — Búsqueda semántica con embeddings + HNSW

**¿Qué es un embedding?** Es una representación numérica del *significado* de un texto. El modelo `all-MiniLM-L6-v2` convierte la descripción de cada empresa en un vector de 384 números. Textos con significado similar → vectores cercanos en ese espacio de 384 dimensiones.

**¿Qué es HNSW?** (Hierarchical Navigable Small World) Es un índice que permite encontrar los vectores más similares sin revisar todos. En lugar de O(N) comparaciones lineales, hace O(log N) saltos por una red jerárquica de vecinos. Sobre 1M de vectores, el speedup es de ~1000×.

**Pipeline completo:**
1. `e.description` (texto) → `all-MiniLM-L6-v2` → vector de 384 floats
2. Vector guardado en NopalDB via `add_node_embedding(node_id, vector, "minilm")`
3. DB cerrada y reabierta → HNSW reconstruido con todos los vectores
4. `similar_to(e, "Atlas Fiduciary Holdings", "minilm")` → HNSW devuelve los 5 más cercanos

---

**¿Qué son los archivos `.npz` en `tutorials/precomputed/`?**

`.npz` es el formato binario comprimido de NumPy (NumPy Zip). Cada archivo guarda dos arrays:

| Array | Forma | Tipo | Contenido |
|-------|-------|------|-----------|
| `texts` | `(N,)` | object | Los textos originales que se embebieron |
| `vectors` | `(N, 384)` | float32 | Los vectores producidos por el modelo |

Los archivos relevantes para este tutorial:
- **`acto2_offshore_minilm.npz`** (~131 KB) — 81 descripciones de `OffshoreEntity` + `ShellCompany`
- **`acto2_offshore_pathref.npz`** (~4 KB) — 2 textos de baseline para `path_anomaly_score`

**¿Por qué existen?** Para evitar descargar el modelo (~90 MB) y recomputar en cada corrida. La primera vez que corres el notebook, el helper `encode_texts()` computa los vectores y los guarda en disco. Las siguientes corridas cargan el `.npz` directamente — sin red, sin modelo en memoria, instantáneo.

**¿Qué pasa si el dataset cambia?** El helper compara los textos del caché con los actuales. Si no coinciden, borra el caché y recomputa. Nunca usará vectores de textos incorrectos. Para forzar recomputo: `encode_texts(..., force_recompute=True)` o borra el `.npz` manualmente.

**Primera corrida:** descarga el modelo (~90 MB, ~1-2 min). **Corridas siguientes:** carga el `.npz` — instantáneo.

In [ ]:
items = []
for label in ("OffshoreEntity", "ShellCompany"):
    res = graph.execute_nql(f"find e.id, e.description from (e:{label})")
    rows = getattr(res, "query", res)
    items.extend(
        (r.get("id", r.get("e.id")), r.get("descr", r.get("e.description")))
        for r in rows
    )
print(f"entidades con descripcion: {len(items)}")

In [ ]:
n = attach_node_embeddings(
    graph,
    items,
    cache_name="acto2_offshore_minilm",
    model_label="minilm",
)
print(f"embeddings adjuntados: {n}")

In [ ]:
# Después de guardar los embeddings en Sled, cerramos y reabrimos el grafo.
# Esto es necesario porque el índice HNSW en memoria se construye al abrir la DB:
# si añadimos vectores mientras el grafo ya está abierto, el índice no los ve
# hasta que se reinicializa. Sin este paso, similar_to devuelve vacío.
if n > 0:
    graph.close()
    del graph  # libera el Arc<RustGraph> inmediatamente en CPython, soltando el lock de Sled
    graph = nopaldb.Graph.open(DB)
    print(f"Grafo reabierto — {n} vectores disponibles en el índice HNSW.")
else:
    print("Sin embeddings: la consulta similar_to será omitida.")

In [ ]:
import pandas as pd
if n > 0:
    df_hnsw = execute_to_df(graph, load_nql("acto_2", "03_hnsw.nql"))
else:
    df_hnsw = pd.DataFrame(columns=["e.name", "e.industry", "e.shell"])
    print("HNSW omitido: add_node_embedding no disponible en este entorno.")
df_hnsw

**Cómo leer el resultado:**
- Los 5 resultados son las entidades cuyas descripciones el modelo consideró semánticamente más cercanas a "Atlas Fiduciary Holdings". **El orden de filas es el orden de scan de Sled (por UUID), no por similitud** — sin `ORDER BY`, "Atlas Fiduciary Holdings" puede aparecer en cualquier posición aunque sea el más cercano (self-match).
- El gate de la sección 8 usa `LIMIT 1` para obtener solo el top-1 de HNSW, lo que sí garantiza el self-match en la única fila del resultado.
- **Las 4-5 entidades restantes** tienen descripciones semánticamente parecidas. El modelo agrupa por señales de industria ("private banking", "commodities", "Harbor Cay") más que por estructura legal.
- **Uso en fraud detection real:** combinarías HNSW con filtros estructurales ("dame entidades similares a Atlas Fiduciary Group que además estén registradas en jurisdicciones de alto riesgo"). El Paso 4 introduce esa dimensión estructural.

## 5. Paso 4 — Path queries cuantificadas: seguir la cadena de control

**¿Qué es un "path" en un grafo?** Es una secuencia de nodos conectados por aristas. En este dataset, un path tipo `A → B → C` significa "A controla a B, que a su vez controla a C".

**¿Qué es un cuantificador `{1,3}`?** Le dice al motor "busca conexiones de longitud mínima 1 y máxima 3". Sin cuantificadores tendrías que escribir 3 consultas separadas (una para cada nivel) y unir los resultados. Con `{1,3}` lo haces en una sola pasada, y NopalDB garantiza *simple-path semantics*: no revisa el mismo nodo dos veces en el mismo camino, evitando bucles infinitos.

**Topología del dataset:** el generador plantó una pirámide determinista. `Atlas Fiduciary Holdings` tiene 3 hijos directos (1-hop), cada hijo tiene 2 hijos (2-hop), y algunos nietos tienen sus propias conexiones (3-hop). Verás exactamente 12 paths: 3 + 6 + 3.

In [ ]:
df_paths = execute_to_df(graph, load_nql("acto_2", "04_paths_quantifier.nql"))
df_paths

## 6. Paso 5 — `path_sum`: ¿cuánto dinero fluye por cada cadena?

**¿Qué hace `path_sum("amount")`?** Suma el valor de la propiedad `amount` en cada arista a lo largo del camino. Si A→B transfiere 5M y B→C transfiere 2M, el path A→B→C tiene `path_sum = 7M`.

**¿Por qué importa la suma y no solo la longitud?** En investigaciones financieras, las rutas más largas no son necesariamente las más peligrosas — son las que mueven más dinero. Un chain de 3 empresas que transfiere 11.5M es más relevante que un chain de 1 empresa que transfiere 500K.

**Lectura del top-3 (seed=42):**
- `Pinnacle International #038` (3 hops, ~11.5M): la ruta de máxima exposición financiera desde el flagship.
- `Onyx Holdings #047` (3 hops, ~7.7M): segunda ruta por la rama derecha de la pirámide.
- `Beacon International #003` (2 hops, ~7M): más corta pero con un solo `amount` muy alto — 7M en un solo salto directo.

En un caso real, estas serían las rutas que priorizarías para investigación manual.

In [ ]:
df_flujo = execute_to_df(graph, load_nql("acto_2", "05_path_minivm.nql"))
df_flujo

## 7. Paso 6 — PathAnomaly: detectar rutas que no encajan con el patrón normal

**¿Qué es detección de anomalías?** Es encontrar cosas que se desvían de lo "normal" sin tener que etiquetar manualmente qué es fraude. En vez de reglas (si X entonces fraude), defines un *baseline* (así es como se ven los paths normales) y mides cuánto se aleja cada path de ese baseline.

**¿Cómo funciona `path_anomaly_score`?**
1. Defines un "path de referencia" — en este caso una conexión típica desde el flagship.
2. Calculas su embedding: el vector promedio de los nodos y aristas del path.
3. Para cada nuevo path, mides `1 - coseno(path_vec, baseline_vec)`.
4. Score → 1.0: el path es muy diferente al baseline. Score → 0.0: el path es muy parecido.

**¿Para qué sirve en Synthetic Offshore Network?** Si tu baseline es "una transferencia directa de 5M a una entidad de private banking", un path con 3 saltos pasando por jurisdicciones de alto riesgo y montos fragmentados (para evitar reportes) tendría score alto — señal de alerta.

El código siguiente registra ese baseline y calcula los scores para todos los paths.

In [ ]:
import numpy as np

edges_df = execute_to_df(graph, """
find e.id, e.amount
from (a) -[e:CONTROLS]-> (b)
""")
if hasattr(graph, "add_edge_embedding") and hasattr(graph, "add_path_reference_embedding"):
    DIM = 384
    for _, row in edges_df.iterrows():
        amount = float(row.get("amount", row.get("e.amount")) or 0.0)
        edge_id = row.get("id", row.get("e.id"))
        h = abs(hash(edge_id)) % (2**32)
        v = np.random.RandomState(h).normal(0, 1, DIM).astype(np.float32)
        v = v * (np.log1p(amount) / 20.0)
        graph.add_edge_embedding(edge_id, v.tolist(), "edge_minilm")
    print(f"edge embeddings adjuntados: {len(edges_df)}")

    ref_path = execute_to_df(graph, """
find a.description as d1, b.description as d2
from (a:OffshoreEntity) -[:CONTROLS]-> (b)
where a.name = "Atlas Fiduciary Holdings"
limit 1
""")
    if not ref_path.empty:
        texts = [ref_path.iloc[0].get("d1", ref_path.iloc[0].get("a.description")), ref_path.iloc[0].get("d2", ref_path.iloc[0].get("b.description"))]
        vecs = encode_texts(texts, cache_name="acto2_offshore_pathref", model_name="all-MiniLM-L6-v2")
        centroid = vecs.mean(axis=0).astype(np.float32)
        graph.add_path_reference_embedding("baseline_pyramid", "minilm", "edge_minilm", centroid.tolist())
        print("baseline_pyramid registrado")
else:
    print("Nota v0.4.19: PathAnomaly omitido porque el binding Python no expone edge/path embeddings.")


In [ ]:
if hasattr(graph, "add_edge_embedding") and n > 0:
    df_anom = execute_to_df(graph, load_nql("acto_2", "06_path_anomaly.nql"))
else:
    df_anom = pd.DataFrame()
    print("Nota v0.4.19: query PathAnomaly omitida en este binding Python.")
df_anom


## 8. Gate de verificación

**Qué valida este gate:** con `LIMIT 1`, `similar_to(e, "Atlas Fiduciary Holdings", "minilm")` debe retornar exactamente esa misma entidad (self-match — distancia coseno ≈ 0 consigo mismo).

**Por qué usar `LIMIT 1` y no `LIMIT 5` del paso anterior:**
- `LIMIT N` controla el parámetro `k` de búsqueda en HNSW.
- Con `k=1`, el único candidato es el vecino más cercano al embedding de referencia.
- Ese vecino es **el propio nodo** (auto-match exacto, distancia = 0.0).
- Sin `ORDER BY`, los resultados con `LIMIT 5` se entregan en orden de scan de Sled (por UUID), **no por similitud** → "Atlas Fiduciary Holdings" puede aparecer en cualquier posición del DataFrame aunque sea el más cercano.
- Con `LIMIT 1 → k=1`, hay solo un candidato posible: el más similar, que es él mismo → resultado determinista.

Esto verifica que el pipeline completo funciona de extremo a extremo:
1. La descripción textual se convirtió a un vector de 384 números.
2. El vector se guardó en Sled con `add_node_embedding`.
3. El índice HNSW se reconstruyó con esos vectores al reabrir la DB.
4. La consulta NQL encontró al vecino más cercano correctamente.

| Medio | Qué verifica |
|-------|-------------|
| **Notebook** | `LIMIT 1` devuelve `"Atlas Fiduciary Holdings"` (self-match HNSW) |
| **Rust** | `top-1 de path_sum == "Pinnacle International #038"` (flujo máximo) |
| **NDBStudio Web** | pegar `queries/03_hnsw.nql` (requiere que el notebook haya persistido los embeddings) |

In [ ]:
if not df_hnsw.empty:
    # Mostrar todos los resultados del LIMIT 5 para exploración
    name_col = next((c for c in df_hnsw.columns if "name" in c.lower()), df_hnsw.columns[0])
    print("Top-5 entidades similares a 'Atlas Fiduciary Holdings' (orden de scan, no de similitud):")
    for i, nm in enumerate(df_hnsw[name_col].astype(str)):
        print(f"  [{i}] {nm!r}")

    # Gate: LIMIT 1 → k=1 en HNSW → único candidato = el propio nodo (distancia 0)
    # Esto es determinista independientemente del orden de scan de Sled.
    df_gate = execute_to_df(graph, """
find e.name
from (e:OffshoreEntity)
where similar_to(e, "Atlas Fiduciary Holdings", "minilm")
limit 1
""")
    gate_col = next((c for c in df_gate.columns if "name" in c.lower()), df_gate.columns[0]) if not df_gate.empty else "e.name"
    top1 = str(df_gate.iloc[0][gate_col]) if not df_gate.empty else "(sin resultados)"
    print(f"\nGate (LIMIT 1 — k=1 en HNSW): {top1!r}")
    assert top1 == "Atlas Fiduciary Holdings", (
        f"GATE FALLIDO: el self-match con LIMIT 1 debe retornar el propio nodo.\n"
        f"  Esperado : 'Atlas Fiduciary Holdings'\n"
        f"  Obtenido : {top1!r}\n"
        f"  Diagnóstico: si top1 != Atlas Fiduciary Holdings, el embedding no está\n"
        f"    en el índice HNSW. Verifica que cerraste y reabriste el grafo (celda\n"
        f"    anterior) y que attach_node_embeddings imprimió 'embeddings adjuntados: 81'.\n"
        f"  Top-5 obtenidos: {df_hnsw[name_col].astype(str).tolist()}"
    )
    print("GATE OK — self-match valida el pipeline embeddings + HNSW.")
elif n > 0:
    raise AssertionError(
        f"GATE FALLIDO: se cargaron {n} embeddings pero similar_to devolvió vacío.\n"
        f"  Causa probable: el índice HNSW no se reconstruyó correctamente.\n"
        f"  Solución: ejecuta todas las celdas en orden (Kernel → Restart & Run All)."
    )
else:
    assert len(items) == 81, f"Esperaba 81 entidades con descripción, obtuve {len(items)}"
    print("GATE OK (modo estructural) — 81 entidades verificadas. HNSW no disponible en este entorno.")

## Siguiente

[`03_biomedical_owl.ipynb`](03_biomedical_owl.ipynb) — reasoner OWL + SHACL.